In [0]:
scope="secretscope"
key="engdados-secret"
storage_account="stoengdados"
application_id = dbutils.secrets.get(scope, "application-id")
directory_id = dbutils.secrets.get(scope, "tenant-id")
service_credential = dbutils.secrets.get(scope=scope,key=key)

spark.conf.set("fs.azure.account.auth.type.%s.dfs.core.windows.net"%(storage_account), "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.%s.dfs.core.windows.net"%(storage_account), "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.%s.dfs.core.windows.net"%(storage_account), application_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.%s.dfs.core.windows.net"%(storage_account), service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.%s.dfs.core.windows.net"%(storage_account), "https://login.microsoftonline.com/%s/oauth2/token"%(directory_id))

# Testando a configuração

In [0]:
container_name = "lakehouse"
# Get all the files under the ADLS folder and create a list of file paths
file_list = dbutils.fs.ls("abfss://%s@%s.dfs.core.windows.net/bronze"%(container_name,storage_account))

# Read each file and create a DataFrame
for file_path in file_list:
    print(file_path)
    df = spark.read.format("csv").options(inferSchema="true", header="true").load(path=f"{file_path.path}*")
    # You can process the DataFrame or register it as a table here
    # For example, to create a temporary table:
    df.createOrReplaceTempView(file_path.name.removesuffix('.csv').replace('.', ''))


FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/bronze/SalesLT.Address.csv', name='SalesLT.Address.csv', size=62730, modificationTime=1774405998000)
FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/bronze/SalesLT.Customer.csv', name='SalesLT.Customer.csv', size=218834, modificationTime=1774405998000)
FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/bronze/SalesLT.CustomerAddress.csv', name='SalesLT.CustomerAddress.csv', size=37652, modificationTime=1774405998000)
FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/bronze/SalesLT.Product.csv', name='SalesLT.Product.csv', size=1358363, modificationTime=1774406000000)
FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/bronze/SalesLT.ProductCategory.csv', name='SalesLT.ProductCategory.csv', size=3456, modificationTime=1774405998000)
FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/bronze/SalesLT.ProductDescription.csv', name='SalesLT.ProductDe

In [0]:
%sql
SHOW VIEWS

namespace,viewName,isTemporary,isMaterialized
,salesltaddress,true,false
,salesltcustomer,true,false
,salesltcustomeraddress,true,false
,salesltproduct,true,false
,salesltproductcategory,true,false
,salesltproductdescription,true,false
,salesltproductmodel,true,false
,salesltproductmodelproductdescription,true,false
,salesltsalesorderdetail,true,false
,salesltsalesorderheader,true,false


# Read Temp Views, clean the data, load into Spark Dataframes
- Filter Rows
- Rename Columns
- Drop Columns

In [0]:
df_salesltsalesorderdetail = spark.sql("SELECT SalesOrderID, OrderQty, ProductID,UnitPrice, UnitPriceDIscount, LineTotal FROM salesltsalesorderdetail")


In [0]:
df_salesltsalesorderheader = spark.sql("SELECT SalesOrderID, RevisionNumber, OrderDate, DueDate, ShipDate, Status, OnlineOrderFlag, SalesOrderNumber, PurchaseOrderNumber, AccountNumber, CustomerID, ShipToAddressID, BillToAddressID, ShipMethod, SubTotal, TaxAmt, Freight, TotalDue FROM salesltsalesorderheader")


In [0]:
# Randomizing the dates in the OrderDate column since our toy AdventureWorks LT dataset only has one distinct order date.
from pyspark.sql.functions import rand, col, expr
df_salesltsalesorderheader = df_salesltsalesorderheader.drop("OrderDate").withColumn("OrderDate", expr("date_add(current_date()-1000, CAST(rand() * 365 AS INT))"))


In [0]:
df_salesltcustomer = spark.sql("SELECT CustomerID, Title, FirstName, MiddleName, LastName, Suffix, CompanyName, EmailAddress, Phone  FROM salesltcustomer")

In [0]:
df_salesltcustomeraddress = spark.sql("SELECT CustomerID, AddressID, AddressType FROM salesltcustomeraddress")

In [0]:
df_salesltproduct = spark.sql("SELECT ProductID, Name, ProductNumber, Color, StandardCost, ListPrice, Size, Weight, ProductCategoryID FROM salesltproduct")


In [0]:
df_salesltproductcategory = spark.sql("SELECT ProductCategoryID, ParentProductCategoryID, Name FROM salesltproductcategory")

# Write Data Frames into ADLS Silver layer as External Tables


In [0]:
path="abfss://%s@%s.dfs.core.windows.net/silver"%(container_name,storage_account)

tableName="salesOrderHeader"
dbutils.fs.mkdirs("%s/%s"%(path,tableName))
# df_salesltsalesorderheader.write.saveAsTable(tableName, format="delta", mode="overwrite", overwriteSchema = "true", path=path + "/" + tableName + "/")
df_salesltsalesorderheader.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")

In [0]:
tableName="salesOrderDetail"
dbutils.fs.mkdirs("%s/%s"%(path,tableName))
# df_salesltsalesorderdetail.write.saveAsTable(tableName, format="delta", mode="overwrite", overwriteSchema = "true", path=path + "/" + tableName + "/")
df_salesltsalesorderdetail.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")


In [0]:
tableName="salesCustomer"
dbutils.fs.mkdirs("%s/%s"%(path,tableName))
#df_salesltcustomer.write.saveAsTable(tableName, format="delta", mode="overwrite", overwriteSchema = "true", path=path + "/" + tableName + "/")
df_salesltcustomer.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")

In [0]:
tableName="salesCustomerAddress"
dbutils.fs.mkdirs("%s/%s"%(path,tableName))
#df_salesltcustomeraddress.write.saveAsTable(tableName, format="delta", mode="overwrite", overwriteSchema = "true", path=path + "/" + tableName + "/")
df_salesltcustomeraddress.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")


In [0]:
tableName="salesProduct"
dbutils.fs.mkdirs("%s/%s"%(path,tableName))
#df_salesltproduct.write.saveAsTable(tableName, format="delta", mode="overwrite", overwriteSchema = "true", path=path + "/" + tableName + "/")
df_salesltproduct.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")

In [0]:
tableName="salesProductCategory"
dbutils.fs.mkdirs("%s/%s"%(path,tableName))
#df_salesltproductcategory.write.saveAsTable(tableName, format="delta", mode="overwrite", overwriteSchema = "true", path=path + "/" + tableName + "/")
df_salesltproductcategory.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")
